In [1]:
import json
import os
import kagglehub
import importlib
import torch
import random
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import re
import pandas as pd
import numpy as np
import gc
from src.utils import load_indices_from_jsonl
from IPython.display import JSON
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from functools import partial
from src.jsonl_dataset import JsonLDataset
from transformers import AutoTokenizer, TrainingArguments, EvalPrediction
from adapters import AutoAdapterModel, AdapterTrainer
from sklearn.metrics import f1_score, precision_score, recall_score
from adapters.composition import Parallel 

### Setup

In [2]:
path = kagglehub.dataset_download("Cornell-University/arxiv/versions/272")
ds_path = os.path.join(path, 'arxiv-metadata-oai-snapshot.json')
master_ds = JsonLDataset(ds_path)
augmented_ds = JsonLDataset("resources/augmented_index.jsonl")

Indexing dataset at C:\Users\kerem\.cache\kagglehub\datasets\Cornell-University\arxiv\versions\272\arxiv-metadata-oai-snapshot.json... (this may take a minute)
Indexed 2951540 entries.
Indexing dataset at resources/augmented_index.jsonl... (this may take a minute)
Indexed 2951540 entries.


In [111]:
from src.index_based_dataset import IndexBasedDataset
from src.utils import load_indices_from_jsonl

In [112]:
test_indices = load_indices_from_jsonl("resources/test_indices_parent_categories.jsonl", flatten=True, shuffle=True, seed=42)
test_ds = IndexBasedDataset(master_ds, augmented_ds, test_indices)

In [3]:
from src.multiadapter_parent_predictor import MultiAdapterParentPredictor
from src import subcategory_predictor

In [189]:
importlib.reload(subcategory_predictor)

<module 'src.subcategory_predictor' from 'C:\\projects\\personal\\python-notebooks\\interview_irisai\\src\\subcategory_predictor.py'>

In [4]:
from sklearn.preprocessing import MultiLabelBinarizer
def create_mlb(target_classes):
    mlb = MultiLabelBinarizer(classes=target_classes)
    mlb.fit([target_classes])
    return mlb

In [35]:
TARGET_SUB_CATEGORIES = json.load(open("resources/target_sub_classes.json", "r"))
def build_adapter_configs():
    result = []
    for sub_cat_name, classes in TARGET_SUB_CATEGORIES.items():
        result.append({
            "category": sub_cat_name,
            "name": f"{sub_cat_name}_categories_adapter",
            "path": f"./resources/{sub_cat_name}_categories_adapter",
            "mbl": create_mlb(classes)
        })
    return result

In [46]:
parent_predictor = MultiAdapterParentPredictor(adapter_configs=[
    {"path": "./resources/parent_categories_adapter/", "name": "arxiv_parent_categories_classifier"},
    {"path": "./resources/parent_categories_adapter_bucket2/", "name": "arxiv_parent_categories_classifier_bucket2"},
    {"path": "./resources/parent_categories_adapter_bucket3/", "name": "arxiv_parent_categories_classifier_bucket3"}
], mlb=create_mlb(list(TARGET_SUB_CATEGORIES.keys())))

Initializing Multi-Adapter Pipeline on cuda...
  -> Loading arxiv_parent_categories_classifier...
  -> Loading arxiv_parent_categories_classifier_bucket2...


There are adapters available but none are activated for the forward pass.


  -> Loading arxiv_parent_categories_classifier_bucket3...


In [6]:
subc_predictor = subcategory_predictor.SubcategoryPredictor(
    adapter_configs=build_adapter_configs()
)

Initializing Multi-Adapter Pipeline on cuda...
  -> Loading Physics_categories_adapter...
  -> Loading Mathematics_categories_adapter...
  -> Loading Computer Science_categories_adapter...
  -> Loading Quantitative Biology_categories_adapter...
  -> Loading Statistics_categories_adapter...
  -> Loading Quantitative Finance_categories_adapter...
  -> Loading Economics_categories_adapter...
  -> Loading Electrical Engineering and Systems Science_categories_adapter...


In [55]:
sample = random.choice(master_ds)
parent_preds = parent_predictor.predict([sample])[0]['labels']
subc_preds = []

for parent in parent_preds:
    subc_out = subc_predictor.predict([sample], category=parent)
    subc_preds.extend(subc_out[0]["labels"])

print(f"ParentC preds: {parent_preds}")
print(f"-" * 50)
print(f"SubC preds: {subc_preds}")
print("-" * 50)
print(f"Actual: {sample["categories"]}")

ParentC preds: ['Computer Science', 'Statistics']
--------------------------------------------------
SubC preds: ['cs.LG', 'stat.CO', 'stat.ML']
--------------------------------------------------
Actual: stat.ML cs.LG


In [67]:
predictor = E2EPredictor(parent_predictor, subc_predictor)

In [66]:
class E2EPredictor:
    def __init__(self, parent_predictor, subcategory_predictor):
        """
        Args:
            parent_predictor: An instance of MultiAdapterInferencePipeline
            subcategory_predictor: An instance of SubCategoryPredictor
        """
        self.parent_predictor = parent_predictor
        self.subcategory_predictor = subcategory_predictor

    def predict(self, items: list, parent_threshold: float = 0.5, sub_threshold: float = 0.5):
        """
        Runs a tiered prediction: 
        1. Identifies parent categories.
        2. For each identified parent, runs the corresponding sub-category adapter.
        """
        final_results = []

        # 1. Get Parent Predictions for the batch
        # We pass the whole list to the parent predictor for efficiency
        parent_outputs = self.parent_predictor.predict(items, threshold=parent_threshold)

        for i, item in enumerate(items):
            detected_parents = parent_outputs[i]['labels']
            all_subcategories = []

            # 2. For each parent detected, run the specific sub-category adapter
            for parent in detected_parents:
                # Note: We pass [item] as a list to match your predictor's signature
                sub_out = self.subcategory_predictor.predict(
                    [item], 
                    category=parent, 
                    threshold=sub_threshold
                )
                
                # If an adapter existed and returned results, collect the labels
                if sub_out:
                    all_subcategories.extend(sub_out[0]["labels"])

            # 3. Consolidate results for this specific item
            final_results.append({
                "parent_categories": detected_parents,
                "labels": list(set(all_subcategories))
            })

        return final_results

In [115]:
sample

{'master_idx': 1362140,
 'title': 'Viewpoint-Aware Channel-Wise Attentive Network for Vehicle\n  Re-Identification',
 'abstract': '  Vehicle re-identification (re-ID) matches images of the same vehicle across\ndifferent cameras. It is fundamentally challenging because the dramatically\ndifferent appearance caused by different viewpoints would make the framework\nfail to match two vehicles of the same identity. Most existing works solved the\nproblem by extracting viewpoint-aware feature via spatial attention mechanism,\nwhich, yet, usually suffers from noisy generated attention map or otherwise\nrequires expensive keypoint labels to improve the quality. In this work, we\npropose Viewpoint-aware Channel-wise Attention Mechanism (VCAM) by observing\nthe attention mechanism from a different aspect. Our VCAM enables the feature\nlearning framework channel-wisely reweighing the importance of each feature\nmaps according to the "viewpoint" of input vehicle. Extensive experiments\nvalidate th

In [174]:
sample = random.choice(test_ds)
res = predictor.predict([sample])[0]

print(f"Predicted Parent categories: {res["parent_categories"]}")
print(f"-" * 50)
print(f"Actual Parent categories: {sample["parent_categories"]}")
print(f"-" * 50)
print(f"Predicted Sub categories: {res["labels"]}")
print("-" * 50)
print(f"Actual Sub categories: {sample["raw_categories"]}")

Predicted Parent categories: ['Mathematics']
--------------------------------------------------
Actual Parent categories: ['Mathematics', 'Physics', 'Mathematics']
--------------------------------------------------
Predicted Sub categories: ['math.PR']
--------------------------------------------------
Actual Sub categories: math.PR math-ph math.MP


### Subc_predictor Test

In [32]:
sample = random.choice(master_ds)
res = subc_predictor.predict([sample], category="Mathematics", threshold=0.8)

print(f"Predicted: {res[0]["labels"]}")
print("-" * 50)
print(f"Actual: {sample["categories"]}")
print("-" * 50)
print(json.dumps(sample, indent=4))

Predicted: []
--------------------------------------------------
Actual: cs.CR
--------------------------------------------------
{
    "id": "1907.03627",
    "submitter": "Gewu Bu",
    "authors": "Gewu Bu (SU), Thanh Son Lam Nguyen (SU), Maria Potop-Butucaru (SU),\n  Kim Thai (SU)",
    "title": "HyperPubSub: Blockchain based Publish/Subscribe",
    "comments": null,
    "journal-ref": null,
    "doi": null,
    "report-no": null,
    "categories": "cs.CR",
    "license": "http://arxiv.org/licenses/nonexclusive-distrib/1.0/",
    "abstract": "  In this paper we describe the architecture and the implementation of a broker\nbased publish/subscribe system where the broker role is played by a private\nblockchain, Hy-perledger Fabric. We show the effectiveness of our architecture\nby implementing and deploying a photo trading plateform. Interestingly, our\narchitecture is generic enough to be adapted to any digital asset trading.\n",
    "versions": [
        {
            "version": "v1